<a href="https://colab.research.google.com/github/lucashobbs17/Geostorm/blob/main/notebooks/01_flare_cme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
   !pip install -q "sunpy[net]"

   from google.colab import drive
   drive.mount('/content/drive')

   import os
   import pandas as pd
   import numpy as np
   import matplotlib.pyplot as plt

   DATA = '/content/drive/MyDrive/geostorm/data'
   os.makedirs(DATA, exist_ok=True)
   print("Setup done")

Mounted at /content/drive
Setup done


   ## 2. Get flares (GOES via HEK)

In [4]:
from sunpy.net import Fido, attrs as a

path = f'{DATA}/flares_2011.csv'

if os.path.exists(path):
    flares = pd.read_csv(path)
    print("Loaded from Drive")
else:
    res = Fido.search(
        a.Time("2011-01-01", "2011-12-31"),
        a.hek.EventType("FL"),
        a.hek.FL.GOESCls > "M1.0",
        a.hek.OBS.Observatory == "GOES",
    )
    cols = ["event_starttime", "event_peaktime", "event_endtime",
            "fl_goescls", "hgs_x", "hgs_y"]
    flares = res["hek"][cols].to_pandas()
    flares.to_csv(path, index=False)
    print("Downloaded and saved")

flares.head()

Loaded from Drive


,event_starttime,event_peaktime,event_endtime,fl_goescls,hgs_x,hgs_y
0,2011-01-28 00:44:00,2011-01-28 01:03:00,2011-01-28 01:10:00,M1.3,0,0
1,2011-02-09 01:23:00,2011-02-09 01:31:00,2011-02-09 01:35:00,M1.9,72,17
2,2011-02-13 17:28:00,2011-02-13 17:38:00,2011-02-13 17:47:00,M6.6,-4,-20
3,2011-02-14 17:20:00,2011-02-14 17:26:00,2011-02-14 17:32:00,M2.2,18,56
4,2011-02-15 01:44:00,2011-02-15 01:56:00,2011-02-15 02:06:00,X2.2,0,0


In [5]:
print(flares.shape)
flares["fl_goescls"].value_counts()

(109, 6)


,count
fl_goescls,
M1.1,12
M1.2,10
M1.3,9
M1.4,8
M1.9,5
M1.6,5
M1.5,5
M1.8,4
M1.7,4


In [6]:
missing = (flares.hgs_x == 0) & (flares.hgs_y == 0)
print("Missing locations:", missing.sum())

dupes = flares[flares.duplicated("event_peaktime", keep=False)]
print("Duplicate rows:", len(dupes))
dupes

Missing locations: 44
Duplicate rows: 2


,event_starttime,event_peaktime,event_endtime,fl_goescls,hgs_x,hgs_y
52,2011-09-06 22:12:00,2011-09-06 22:20:00,2011-09-06 22:24:00,X2.1,18,14
53,2011-09-06 22:12:00,2011-09-06 22:20:00,2011-09-06 22:24:00,X2.1,18,14


In [7]:
flares.loc[missing, ["hgs_x", "hgs_y"]] = np.nan
flares = flares.drop_duplicates("event_peaktime").reset_index(drop=True)
print(flares.shape)

(108, 6)


In [8]:
for c in ["event_starttime", "event_peaktime", "event_endtime"]:
    flares[c] = pd.to_datetime(flares[c])

flares = flares.sort_values("event_peaktime").reset_index(drop=True)

In [9]:
path_ssw = f'{DATA}/ssw_locations_2011.csv'

if os.path.exists(path_ssw):
    ssw = pd.read_csv(path_ssw)
else:
    res2 = Fido.search(
        a.Time("2011-01-01", "2011-12-31"),
        a.hek.EventType("FL"),
        a.hek.FRM.Name == "SSW Latest Events",
    )
    ssw = res2["hek"][["event_peaktime", "hgs_x", "hgs_y"]].to_pandas()
    ssw.to_csv(path_ssw, index=False)

ssw["event_peaktime"] = pd.to_datetime(ssw["event_peaktime"])
ssw = ssw.sort_values("event_peaktime").reset_index(drop=True)
print(ssw.shape)

(2546, 3)


In [10]:
merged = pd.merge_asof(
    flares,
    ssw.rename(columns={"hgs_x": "ssw_x", "hgs_y": "ssw_y"}),
    on="event_peaktime",
    direction="nearest",
    tolerance=pd.Timedelta("10min"),
)

flares["hgs_x"] = flares["hgs_x"].fillna(merged["ssw_x"])
flares["hgs_y"] = flares["hgs_y"].fillna(merged["ssw_y"])
print("Still missing:", flares["hgs_x"].isna().sum())

Still missing: 7


In [12]:
flares[flares.event_peaktime.dt.date == pd.Timestamp("2011-02-15").date()]
print("Zero locations:", ((flares.hgs_x == 0) & (flares.hgs_y == 0)).sum())
zero = (flares.hgs_x == 0) & (flares.hgs_y == 0)
flares.loc[zero, ["hgs_x", "hgs_y"]] = np.nan

flares = flares.dropna(subset=["hgs_x", "hgs_y"]).reset_index(drop=True)
flares.to_csv(f'{DATA}/flares_2011_clean.csv', index=False)
print(flares.shape)

Zero locations: 1
(100, 6)


## 3. Get CMEs (CDAW LASCO catalogue)


In [13]:
import requests

path_cme = f'{DATA}/cdaw_univ_all.txt'

if not os.path.exists(path_cme):
    url = "https://cdaw.gsfc.nasa.gov/CME_list/UNIVERSAL/text_ver/univ_all.txt"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(path_cme, "w") as f:
        f.write(r.text)
    print("Downloaded")
else:
    print("Already saved")

Downloaded


In [15]:
rows = []
with open(path_cme) as f:
    for line in f:
        parts = line.split()
        if len(parts) >= 5 and "/" in parts[0] and parts[0][:4].isdigit():
            rows.append(parts[:5])

cmes = pd.DataFrame(rows, columns=["date", "time", "cpa", "width", "speed"])
cmes["cme_time"] = pd.to_datetime(cmes["date"] + " " + cmes["time"])
cmes["halo"] = cmes["cpa"] == "Halo"

for c in ["cpa", "width", "speed"]:
    cmes[c] = pd.to_numeric(cmes[c], errors="coerce")

cmes = cmes[cmes["cme_time"].dt.year == 2011].reset_index(drop=True)
cmes = cmes.drop(columns=["date", "time"])

In [16]:
print(cmes.shape)
print("Halo CMEs:", cmes["halo"].sum())
cmes.head()

(1991, 5)
Halo CMEs: 41


,cpa,width,speed,cme_time,halo
0,49.0,18,251.0,2011-01-01 07:48:05,False
1,262.0,48,253.0,2011-01-02 06:12:05,False
2,268.0,101,355.0,2011-01-02 15:24:05,False
3,126.0,13,225.0,2011-01-03 07:12:05,False
4,13.0,35,154.0,2011-01-04 12:48:05,False


## 4. Match flares to CMEs (labels)

In [17]:
lon = np.radians(flares["hgs_x"])
lat = np.radians(flares["hgs_y"])

x = np.cos(lat) * np.sin(lon)   # east-west on the disk (west positive)
y = np.sin(lat)                 # north-south on the disk

flares["flare_pa"] = np.degrees(np.arctan2(-x, y)) % 360
flares["dist_from_centre"] = np.sqrt(x**2 + y**2)

flares[["fl_goescls", "hgs_x", "hgs_y", "flare_pa", "dist_from_centre"]].head()

,fl_goescls,hgs_x,hgs_y,flare_pa,dist_from_centre
0,M1.3,88.0,16.0,286.009253,0.999437
1,M1.9,72.0,17.0,287.820742,0.955338
2,M6.6,-4.0,-20.0,169.150577,0.348245
3,M2.2,18.0,56.0,348.226151,0.846855
4,M1.1,29.0,-19.0,234.616387,0.562247


In [23]:
def angle_diff(a, b):
    d = abs(a - b) % 360
    return np.minimum(d, 360 - d)

WINDOW = pd.Timedelta("60min")
MAX_ANGLE = 45

def match_flares(flares, cmes, shift=pd.Timedelta(0)):
    labels = []
    for _, f in flares.iterrows():
        start = f.event_starttime + shift
        cand = cmes[(cmes.cme_time >= start) &
                    (cmes.cme_time <= start + WINDOW)]
        ok = cand[cand.halo | (angle_diff(cand.cpa, f.flare_pa) <= MAX_ANGLE)]
        labels.append(int(len(ok) > 0))
    return np.array(labels)

flares["cme"] = match_flares(flares, cmes)
print(flares["cme"].value_counts())


cme
0    65
1    35
Name: count, dtype: int64


In [25]:
flares.to_csv(f'{DATA}/flares_2011_labelled.csv', index=False)

Realised benefit from 60-90 produced as much noise as real flares, so the window was shortened


In [20]:
for h in [-24, -12, 12, 24]:
    fake = match_flares(flares, cmes, pd.Timedelta(hours=h))
    print(f"Shift {h:+}h: {fake.mean():.0%} matched")

Shift -24h: 20% matched
Shift -12h: 11% matched
Shift +12h: 15% matched
Shift +24h: 16% matched


In [21]:
shifts = [pd.Timedelta(hours=h) for h in [-24, -12, 12, 24]]

for w in [45, 60, 90]:
    for ang in [30, 45]:
        WINDOW = pd.Timedelta(minutes=w)
        MAX_ANGLE = ang
        real = match_flares(flares, cmes).mean()
        chance = np.mean([match_flares(flares, cmes, s).mean() for s in shifts])
        print(f"{w:>3} min, {ang}°: real {real:.0%}, chance {chance:.0%}, "
              f"gap {real - chance:.0%}")

WINDOW = pd.Timedelta("90min")   # reset to the originals
MAX_ANGLE = 45

 45 min, 30°: real 26%, chance 6%, gap 20%
 45 min, 45°: real 28%, chance 7%, gap 21%
 60 min, 30°: real 33%, chance 8%, gap 25%
 60 min, 45°: real 35%, chance 9%, gap 26%
 90 min, 30°: real 38%, chance 15%, gap 24%
 90 min, 45°: real 43%, chance 16%, gap 28%
